In [ ]:
import numpy as np
import pandas as pd
import json
import yaml
from tqdm import tqdm

In [ ]:
from openai import OpenAI
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
raw_data = pd.read_csv(dataset_config['path_processed'] + 'CNKI/06_CNKI_DID.csv')
raw_data.head()

In [ ]:
df_sum = (
    raw_data
      .groupby('corporate_id', as_index=False)
      .agg(
          corporate=('corporate', 'first'), 
          count=('pub_num', 'sum') 
      )
)

geo_db = df_sum[['corporate', 'count']].sort_values(by='count', ascending=False).reset_index(drop=True)
geo_db

In [ ]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=openai_api_key,
)

response = client.responses.create(
    model="gpt-4o-mini",
    instructions="Determine whether the given Chinese firm is state-owned (return 1) or privately owned (return 0). If unsure, return NaN. Only return 1, 0, or NaN — no additional information.",
    input="中国海油-C",
)

print(response.output_text)

In [ ]:
def create_and_upload_batch(sample_df, client, batch_desc, model="gpt-4o-mini"):
    # Step 1: Create the JSONL file
    batch_filename = dataset_config['path_processed'] + '_CHAT/CNKI_UPLOAD_' + batch_desc + '.jsonl'
    print(batch_filename)
    with open(batch_filename, 'w', encoding='utf-8') as f:
        for idx, row in sample_df.iterrows():
            request_data = {
                "custom_id": f"request-{idx}",
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": model,
                    "instructions": ("Determine whether the given Chinese firm is state-owned (return 1) or privately owned (return 0). If unsure, return NaN. Only return 1, 0, or NaN — no additional information."),
                    "input": f"{row['corporate']}"
                }
            }
            f.write(json.dumps(request_data) + "\n")

    # Step 2: Upload the JSONL file using the Files API
    with open(batch_filename, "rb") as f:
        upload_response = client.files.create(
            file=f,
            purpose="batch"
        )
    input_file_id = upload_response.id
    print("Uploaded file ID:", input_file_id)
    
    # Step 3: Create the batch job
    upload_batch = client.batches.create(
        input_file_id=input_file_id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "description": 'Check country: ' + batch_desc
        }
    )
    batch_id = upload_batch.id
    print("Uploaded batch ID:", batch_id)
    print()
    
    return batch_id

In [ ]:
use_larger_model = ['over200']
name_larger_model = 'gpt-4o'

geo_db_upload = {}
geo_db_upload['over200'] = geo_db[geo_db['count'].apply(lambda x: x >= 200)]
geo_db_upload['10to199'] = geo_db[geo_db['count'].apply(lambda x: 1 < x < 200)]

In [ ]:
def split_large_dfs(dfs_dict, max_rows=40000):
    new_dict = {}
    for key, df in dfs_dict.items():
        n_rows = len(df)
        if n_rows > max_rows:
            # Calculate the number of chunks needed using ceiling division
            num_chunks = (n_rows + max_rows - 1) // max_rows
            for i in range(num_chunks):
                new_key = f"{key}_{i+1}"
                new_dict[new_key] = df.iloc[i*max_rows:(i+1)*max_rows]
        else:
            new_dict[key] = df
    return new_dict

In [ ]:
db_upload = split_large_dfs(geo_db_upload)
[f'{k}: {len(v)}'for k,v in db_upload.items()]

In [ ]:
batch_ids = {}
for batch_desc in db_upload:
    if batch_desc in use_larger_model:
        batch_ids[batch_desc] = create_and_upload_batch(db_upload[batch_desc], client, batch_desc, name_larger_model)
    else:
        batch_ids[batch_desc] = create_and_upload_batch(db_upload[batch_desc], client, batch_desc, 'gpt-4o-mini')

In [ ]:
batch_ids

In [ ]:
batch_ids = {'over200': 'batch_6836d300e8f88190bb193913eb20160f',
             '10to199_1': 'batch_6836d31d51308190b6cdc4ddc56e10ce',
             '10to199_2': 'batch_6836d327cd788190aed6c4865a22f670'} # change this

In [ ]:
# Check batch status
for task_name in batch_ids:
    batch_id = batch_ids[task_name]
    batch = client.batches.retrieve(batch_id)
    print(f'{task_name:18s}', batch.id, batch.status, batch.request_counts, sep='\t')

In [ ]:
# Get batch outputs
for task_name in batch_ids:
    batch_id = batch_ids[task_name]
    batch = client.batches.retrieve(batch_id)
    print(f'{task_name:18s}', batch.status)
    if batch.status == 'completed':
        print('  - Getting:', task_name)
        output_response = client.files.content(batch.output_file_id)
        output_content = output_response.read().decode("utf-8")
        batch_filename = dataset_config['path_processed'] + '_CHAT/CNKI_OUT_' + task_name + '.jsonl'
        with open(batch_filename, 'w', encoding='utf-8') as f:
            f.write(output_content)

In [ ]:
# Get batch outputs
batch_outputs = {}
for task_name in batch_ids:
    batch_filename = dataset_config['path_processed'] + '_CHAT/CNKI_OUT_' + task_name + '.jsonl'
    with open(batch_filename, 'r', encoding='utf-8') as f:
        batch_outputs[task_name] = f.read()

In [ ]:
# Process each line to map custom_ids to their outputs.
batch_results = {}
for task_name in batch_outputs:
    batch_filename = dataset_config['path_processed'] + '_CHAT/CNKI_UPLOAD_' + task_name + '.jsonl'

    # Step 1: Read the JSONL file and build a mapping from index to input.
    input_mapping = {}
    with open(batch_filename, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # Extract the index from custom_id (e.g., "request-3" -> 3)
            idx = int(data.get("custom_id").split('-')[1])
            input_value = data["body"]["input"]
            input_mapping[idx] = input_value

    # Step 2: Process your batch outputs to build a mapping from index to output text.
    results = {}
    output_content = batch_outputs[task_name]
    for line in output_content.strip().split("\n"):
        data = json.loads(line)
        idx = int(data.get("custom_id").split('-')[1])
        output_text = data['response']['body']['output'][0]['content'][0]['text']
        results[idx] = output_text

    # Step 3: Merge the input and output mappings into a DataFrame for further processing or review.
    # This DataFrame will have one row per request, with the original input and the corresponding output.
    df_result = pd.DataFrame({
        'corporate': [input_mapping[idx] for idx in sorted(input_mapping.keys())],
        'SOE_ChatGPT': [results[idx] for idx in sorted(results.keys())]
    })
    
    batch_results[task_name] = df_result

In [ ]:
result_merged = pd.concat(batch_results.values(), ignore_index=True)
result_merged

In [ ]:
# check
row = result_merged.loc[result_merged['corporate'] == '中国海油-C']
row

In [ ]:
full_merged = raw_data.merge(result_merged, on='corporate', how='left')
full_merged.head()

In [ ]:
full_merged.to_csv(dataset_config['path_processed'] + 'CNKI/07_CNKI_SOE_ChatGPT.csv', index=False, encoding='utf-8')